# Semifreddo vs Full Akita — Benchmark

Inserts a strong CTCF motif into the central bin of each flat-region sequence,
then compares full Akita vs Semifreddo on:
- **Prediction accuracy**: Pearson R
- **Runtime**: wall-clock time per forward pass
- **Memory**: peak GPU memory allocated

All metrics are computed for every pair in the table.

```
center bin (in map) : 256
center bin (in 640) : 256 + 64 (cropping) = 320
sequence window     : bins 251–261 → 11 bins × 2048 bp
bins replaced       : 318–322 (central 5)
```

## 1. Imports

In [1]:
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))
sys.path.insert(0, os.path.abspath("/home1/smaruj/ledidi_akita/"))

from akita.model import SeqNN
from semifreddo.semifreddo import SemifreddoLedidiWrapper
from utils.data_utils import one_hot_encode_sequence
from pyfaidx import Fasta

## 2. Parameters

In [2]:
FOLD = 0

# Central bin indices
CENTER_BIN_MAP = 256                          # in 512-bin contact map
CENTER_BIN_640 = CENTER_BIN_MAP + 64          # in 640-bin tower output = 320
BIN_SIZE       = 2048                         # bp per bin
CONTEXT_BINS   = 5                            # ±5 bins padding for Semifreddo tower

SEQ_LEN         = 640 * BIN_SIZE          # 1,310,720
CENTER_BP_START = SEQ_LEN // 2            # 655,360
CENTER_BP_END   = CENTER_BP_START + BIN_SIZE  # 657,408

# Paths
GENOME_PATH = "/project2/fudenber_735/genomes/mm10/mm10.fa"
MODEL_PATH  = ("/home1/smaruj/pytorch_akita/models/finetuned/mouse/Hsieh2019_mESC/checkpoints/"
               "Akita_v2_mouse_Hsieh2019_mESC_model0_finetuned.pth")
FLAT_TSV    = (f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/"
               f"analysis/flat_regions/mouse_flat_regions_chrom_states_tsv/"
               f"fold{FOLD}_selected_genomic_windows_centered_chrom_states.tsv")
CTCF_CSV    = "/home1/smaruj/ledidi_akita/data/ctcf_tables/top100_strong_ctcf_by_insertion_score_mESC.csv"
SEQ_DIR     = (f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/"
               f"analysis/flat_regions/mouse_sequences")
TOWER_DIR   = (f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/"
               f"analysis/flat_regions/mouse_tower_outputs")

## 3. Load Model, Genome, and Tables

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = SeqNN()
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.to(device).eval()
print("Model loaded")

# %%
genome  = Fasta(GENOME_PATH)
flat_df = pd.read_csv(FLAT_TSV, sep="\t")
ctcf_df = pd.read_csv(CTCF_CSV)

Device: cuda:0
Model loaded


## 4. Build Cartesian Product (flat regions × CTCF motifs)

In [4]:
def get_ctcf_forward_seq(chrom, start, end, strand):
    seq = genome[chrom][start:end].seq
    if strand == "-":
        complement = str.maketrans("ACGTacgt", "TGCAtgca")
        seq = seq[::-1].translate(complement)
    return seq.upper()

ctcf_df["ctcf_seq"] = ctcf_df.apply(
    lambda r: get_ctcf_forward_seq(r["chrom"], r["start"], r["end"], r["strand"]), axis=1
)

In [ ]:
flat_df["_key"] = 1
ctcf_df["_key"] = 1
merged_df = (
    pd.merge(flat_df, ctcf_df, on="_key", suffixes=("_flat", "_ctcf"))
    .drop(columns="_key")
    .rename(columns={
        "chrom_flat": "chrom",
        "chrom_ctcf": "ctcf_chrom",
        "start":      "ctcf_start",
        "end":        "ctcf_end",
        "strand":     "ctcf_strand",
    })
    .reset_index(drop=True)
)
print(f"Total pairs: {len(merged_df):,}")

## 5. Helper Functions

In [ ]:
def load_and_insert_ctcf(row):
    X = torch.load(
        f"{SEQ_DIR}/{row['fold']}/{row['chrom']}_{row['centered_start']}_{row['centered_end']}_X.pt",
        weights_only=True,
    )
    ctcf_ohe = one_hot_encode_sequence(row["ctcf_seq"])  # (1, 4, 19)
    ctcf_len = ctcf_ohe.shape[2]
    insert_start = CENTER_BP_START + (BIN_SIZE - ctcf_len) // 2
    insert_end   = insert_start + ctcf_len
    X[0, :, insert_start:insert_end] = torch.tensor(ctcf_ohe[0], dtype=X.dtype)
    return X


def load_tower(row):
    return torch.load(
        f"{TOWER_DIR}/{row['fold']}/{row['chrom']}_{row['centered_start']}_{row['centered_end']}_tower_out.pt",
        weights_only=True,
    ).to(device)  # (1, 128, 640)


def build_sf_wrapper(model, X, tower):
    return SemifreddoLedidiWrapper(
        model                   = model,
        precomputed_full_output = tower,
        full_X                  = X,
        edited_bin              = CENTER_BIN_MAP,
        context_bins            = CONTEXT_BINS,
        cropping_applied        = 64,
    )

## 6. Run Benchmark (all pairs)

In [ ]:
pred_times_seqnn = []
pred_mem_seqnn   = []
pred_times_sf    = []
pred_mem_sf      = []
pearson_rs       = []

for idx, row in merged_df.iterrows():
    print(idx)

    X     = load_and_insert_ctcf(row).to(device)
    tower = load_tower(row)

    sf_wrapper = build_sf_wrapper(model, X, tower)
    X_center   = X[:, :, sf_wrapper.center_bp_start:sf_wrapper.center_bp_end]  # (1, 4, 2048)
    
    # Full Akita
    torch.cuda.reset_peak_memory_stats(device)
    t0 = time.time()
    with torch.no_grad():
        pred_full = model(X)
    pred_times_seqnn.append(time.time() - t0)
    pred_mem_seqnn.append(torch.cuda.max_memory_allocated(device) / 1e6)
    
    # Semifreddo
    torch.cuda.reset_peak_memory_stats(device)
    t0 = time.time()
    with torch.no_grad():
        pred_sf = sf_wrapper(X_center)
    pred_times_sf.append(time.time() - t0)
    pred_mem_sf.append(torch.cuda.max_memory_allocated(device) / 1e6)

    # Pearson R
    r, _ = pearsonr(
        pred_full.cpu().flatten().numpy(),
        pred_sf.cpu().flatten().numpy(),
    )
    
    pearson_rs.append(r)

merged_df["pred_time_seqnn"] = pred_times_seqnn
merged_df["pred_mem_seqnn_MB"] = pred_mem_seqnn
merged_df["pred_time_sf"]    = pred_times_sf
merged_df["pred_mem_sf_MB"]  = pred_mem_sf
merged_df["pearsonR"]        = pearson_rs

## 7. Print Summary Metrics

In [ ]:
avg_metrics = merged_df[["pred_time_seqnn", "pred_mem_seqnn_MB",
                          "pred_time_sf",    "pred_mem_sf_MB",
                          "pearsonR"]].mean()
print(avg_metrics.to_string())
print(f"\nSpeedup : {avg_metrics['pred_time_seqnn'] / avg_metrics['pred_time_sf']:.2f}×")
print(f"Memory  : {avg_metrics['pred_mem_seqnn_MB'] / avg_metrics['pred_mem_sf_MB']:.2f}× reduction")

## 8. Memory Usage Plot (per-layer, averaged across all runs)

In [ ]:
memory_log_full = []
memory_log_sf   = []

In [ ]:
def make_hook(log):
    def hook(module, inp, out, name=""):
        if torch.cuda.is_available():
            log.append({
                "layer":        f"{name}: {module.__class__.__name__}",
                "allocated_MB": torch.cuda.memory_allocated() / 1e6,
                "peak_MB":      torch.cuda.max_memory_allocated() / 1e6,
            })
    return hook

In [ ]:
all_logs_full = []
all_logs_sf   = []

for idx, row in merged_df.iterrows():
    X     = load_and_insert_ctcf(row).to(device)
    tower = load_tower(row)
    sf_wrapper = build_sf_wrapper(model, X, tower)
    X_center   = X[:, :, sf_wrapper.center_bp_start:sf_wrapper.center_bp_end]

    # Full Akita memory log
    memory_log_full.clear()
    handles = [
        m.register_forward_hook(make_hook(memory_log_full).__func__ if False else
                                lambda m, i, o, n=name: memory_log_full.append({
                                    "layer": f"{n}: {m.__class__.__name__}",
                                    "allocated_MB": torch.cuda.memory_allocated() / 1e6,
                                    "peak_MB": torch.cuda.max_memory_allocated() / 1e6,
                                }))
        for name, m in model.named_modules()
    ]
    with torch.no_grad():
        model(X)
    [h.remove() for h in handles]
    df_log = pd.DataFrame(memory_log_full)
    df_log["run"] = idx
    all_logs_full.append(df_log)

    # Semifreddo memory log
    memory_log_sf.clear()
    handles = [
        m.register_forward_hook(
            lambda m, i, o, n=name: memory_log_sf.append({
                "layer": f"{n}: {m.__class__.__name__}",
                "allocated_MB": torch.cuda.memory_allocated() / 1e6,
                "peak_MB": torch.cuda.max_memory_allocated() / 1e6,
            }))
        for name, m in model.named_modules()
    ]
    with torch.no_grad():
        sf_wrapper(X_center)
    [h.remove() for h in handles]
    df_log = pd.DataFrame(memory_log_sf)
    df_log["run"] = idx
    all_logs_sf.append(df_log)

### Average memory per layer and plot

In [ ]:
df_all_full = pd.concat(all_logs_full, ignore_index=True)
df_all_sf   = pd.concat(all_logs_sf,   ignore_index=True)

df_avg_full = df_all_full.groupby("layer")[["allocated_MB", "peak_MB"]].mean().reset_index()
df_avg_sf   = df_all_sf.groupby("layer")[["allocated_MB", "peak_MB"]].mean().reset_index()

# Preserve forward-pass layer order
ordered_layers = [
    f"{name}: {m.__class__.__name__}"
    for name, m in model.named_modules() if name != ""
]
set_full = set(df_avg_full["layer"])
set_sf   = set(df_avg_sf["layer"])
common   = [l for l in ordered_layers if l in set_full and l in set_sf]

df_avg_full = df_avg_full.set_index("layer").reindex(common).reset_index()
df_avg_sf   = df_avg_sf.set_index("layer").reindex(common).reset_index()

plt.figure(figsize=(20, 3))
x = range(len(common))
plt.plot(x, df_avg_full["allocated_MB"], marker="o", label="Akita")
plt.plot(x, df_avg_sf["allocated_MB"], marker="o", label="Semifreddo")
plt.xlabel("Layer index (forward order)")
plt.ylabel("Allocated memory (MB)")
plt.title(f"Average allocated memory per layer (n={len(merged_df)} runs)")
plt.legend()
plt.tight_layout()
# plt.savefig("./plots/semifreddo_full_akita_memory.svg", format="svg")
plt.show()